# Beginner's guide to `gemma-ast-agent`
### Every function, what it means, and why it is there

This notebook explains the code behind the paper **"Localizing Gemma 4 Agents via AST-Guided Retrieval and Self-Correction"** (Kaggle, Google – The Gemma 4 Developer Agent Paper Track).

**Who it is for:** anyone who knows a little Python (functions, lists, dictionaries) and wants to understand how a small AI coding agent works from the inside.

**What you will learn**

1. How the agent turns a codebase into small, searchable pieces (*AST indexing*)
2. How it picks which code to show the model (*retrieval* with BM25 and a call graph)
3. How the model's answer becomes a real code change (*SEARCH/REPLACE edits*)
4. How the agent runs the tests and tries again when they fail (*self-correction*)
5. How we planted 50 bugs to build a fair benchmark (*mutation testing*)
6. How the results and statistics in the paper are computed

**No GPU needed.** Every demo uses a tiny bundled library (`minilib`) and a *fake* model that returns scripted answers, so the whole notebook runs in about a minute on any laptop, Kaggle or Colab. The last part shows how to plug in a real Gemma 4 model.

**How each function is explained.** Every function has a card with the same headings: *What it is* (the definition), *Inputs*, *Returns*, and *Why it exists*. Most cards are followed by a small code cell that runs the function so you can see what it does. Functions whose names start with `_` are internal helpers. They are explained too, because beginners often get stuck exactly there.

**Links:** code: https://github.com/anisha274/gemma-ast-agent · video: https://youtu.be/E8VtsAhGYNA

## Contents

0. Setup
1. The big picture (and a glossary)
2. `indexer.py`: turning code into symbols
3. `retrieval.py`: choosing what the model sees
4. `prompts.py`: what we actually say to the model
5. `models.py`: talking to Gemma (or a fake model)
6. `edits.py`: turning the model's answer into a code change
7. `runner.py`: running the tests safely
8. `agent.py`: the full retrieve → patch → test → retry loop
9. `mutate.py`: building the benchmark by planting bugs
10. `eval.py`: running experiments and computing statistics
11. `results/analyze.py`: regenerating every number in the paper
12. Running with a real Gemma 4 model
13. Cheat sheet: every function in one table

## 0. Setup

This cell finds the project code. If you opened the notebook inside the repository it uses that copy; otherwise (Kaggle, Colab) it downloads the repository from GitHub. It also installs `pytest`, which the demos use to run tests.

In [1]:
import os, sys, subprocess, shutil, tempfile, json

REPO_URL = "https://github.com/anisha274/gemma-ast-agent.git"
if os.path.isdir("ast_agent"):
    ROOT = os.getcwd()
elif os.path.isdir(os.path.join("..", "ast_agent")):
    ROOT = os.path.abspath("..")
else:
    if not os.path.isdir("gemma-ast-agent"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", REPO_URL], check=True)
    ROOT = os.path.abspath("gemma-ast-agent")
sys.path.insert(0, ROOT)

try:
    import pytest  # noqa: F401
except ImportError:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pytest"], check=True)

os.environ["PY_COLORS"] = "0"   # plain pytest output (Jupyter would otherwise add colour codes)

SAMPLE = os.path.join(ROOT, "benchmarks", "sample")
SAMPLE_REPO = os.path.join(SAMPLE, "repo")

def fresh_copy():
    # A throwaway copy of the sample library, so demos never change the original files.
    dst = tempfile.mkdtemp(prefix="minilib_")
    shutil.copytree(SAMPLE_REPO, dst, dirs_exist_ok=True,
                    ignore=shutil.ignore_patterns("__pycache__", ".pytest_cache"))
    return dst

print("Project root:", ROOT)
print("Sample library files:", sorted(os.listdir(os.path.join(SAMPLE_REPO, "minilib"))))

Project root: gemma-ast-agent
Sample library files: ['__init__.py', 'shapes.py', 'stats.py', 'text.py']


Let's look at the sample library we will use everywhere. It is a tiny package with three modules (`stats.py`, `shapes.py`, `text.py`) and a test file. Here is `stats.py`:

In [2]:
print(open(os.path.join(SAMPLE_REPO, "minilib", "stats.py")).read())

"""Small statistics helpers."""

DEFAULT_PRECISION = 2


def mean(values):
    """Arithmetic mean of a non-empty sequence."""
    if len(values) == 0:
        raise ValueError("mean() of empty sequence")
    return sum(values) / len(values)


def _sorted_copy(values):
    return sorted(list(values))


def median(values):
    """Middle value; average of the two middle values for even-length input."""
    data = _sorted_copy(values)
    n = len(data)
    if n == 0:
        raise ValueError("median() of empty sequence")
    mid = n // 2
    if n % 2 == 1:
        return data[mid]
    return (data[mid - 1] + data[mid]) / 2


def variance(values):
    """Population variance."""
    m = mean(values)
    return sum((x - m) ** 2 for x in values) / len(values)


def clamp(x, low, high):
    """Limit x to the closed interval [low, high]."""
    if low > high:
        raise ValueError("low must be <= high")
    if x < low:
        return low
    if x > high:
        return high
    return x


def

## 1. The big picture

A coding agent gets a **bug report** ("these tests fail") and must **edit the code** so the tests pass. A small model like Gemma 4 E4B can only read about **6,000 tokens** of code at once, far less than a real library. So the agent has to decide *what to show it*.

```
            ┌───────────────┐   ┌──────────────┐   ┌──────────────┐   ┌──────────┐
 issue ───► │ 1. INDEX repo │──►│ 2. RETRIEVE  │──►│ 3. ASK MODEL │──►│ 4. APPLY │
            │  (indexer.py) │   │ (retrieval)  │   │  (models.py) │   │ (edits)  │
            └───────────────┘   └──────────────┘   └──────────────┘   └────┬─────┘
                                                                           ▼
                     ┌──────────── tests fail: reset + feedback ──── 5. RUN TESTS
                     │                                               (runner.py)
                     └──► back to step 3 (up to 3 attempts)          tests pass ─► done
```

`agent.py` runs this loop. `mutate.py` builds the benchmark, and `eval.py` runs many tasks and computes the statistics.

### Glossary (plain-English definitions)

| Term | Meaning |
|---|---|
| **AST** (Abstract Syntax Tree) | Python's own structured view of source code: a tree of "function", "class", "if statement" nodes. Python's built-in `ast` module builds it for us. |
| **Symbol** | One retrievable unit of code: a function, a method, a class outline, or a module header. |
| **Token** | The unit a language model reads. For code, roughly 4 characters per token. |
| **Context budget** | The maximum number of tokens of code we put in the prompt (6,000 here). |
| **Retrieval** | Choosing which pieces of code to show the model. |
| **BM25** | A classic search-engine formula that scores how well a document matches a query by counting shared words, favouring rare words. |
| **Call graph** | Who calls whom: `median` calls `_sorted_copy`, so they are neighbours in the graph. |
| **Traceback** | The error report Python prints, listing the file and line of each function involved in a crash. |
| **Patch / diff** | A description of a code change: which lines were removed (`-`) and added (`+`). |
| **SEARCH/REPLACE block** | The format the model uses to describe an edit: the exact old lines, then the new lines. |
| **Self-correction** | When the tests still fail, show the model what went wrong and let it try again. |
| **Mutation** | A tiny deliberate bug, for example changing `<` to `<=`. We plant these to create tasks. |
| **Gold symbol** | The function where the bug really is (known because we planted it). |
| **Oracle** | An experimental condition that cheats on purpose: it always shows the gold function. It tells us how good retrieval could possibly be. |
| **Resolve rate** | The share of tasks where the full test suite passes after the agent's patch. |
| **Wilson interval / McNemar test** | Statistics for "how sure are we?" (Section 10). |

## 2. `indexer.py`: turning code into symbols

**Goal:** split a repository into small, meaningful pieces (functions, methods, classes) with exact line numbers, plus a record of which function calls which. These pieces are what the retriever searches.

First, the module-level constants:

- **`SKIP_DIRS`**: folders that are never indexed (`.git`, virtual environments, build output, caches). They contain no library code, or a copy of someone else's.
- **`TEST_GLOBS`**: filename patterns that identify test files (`tests/*`, `test_*.py`, `conftest.py`, ...). Tests are excluded from retrieval, and later the agent is not allowed to edit them.
- **`MAX_FILE_BYTES`**: files larger than 1 MB are skipped (usually generated data, not hand-written code).

In [3]:
from ast_agent import indexer
print("Skipped folders:", sorted(indexer.SKIP_DIRS)[:6], "...")
print("Test patterns:", indexer.TEST_GLOBS)

Skipped folders: ['.eggs', '.git', '.mypy_cache', '.pytest_cache', '.tox', '.venv'] ...
Test patterns: ('tests/*', '*/tests/*', 'test/*', '*/test/*', 'test_*.py', '*/test_*.py', '*_test.py', 'conftest.py', '*/conftest.py')


### `is_test_path`

```python
def is_test_path(rel_path: str) -> bool
```

**What it is:** Checks whether a file path looks like a test file, by matching it against every pattern in `TEST_GLOBS`.

**Inputs:**
- `rel_path`: a path relative to the repository root, e.g. `"tests/test_minilib.py"`.

**Returns:** `True` if the path is a test file, otherwise `False`.

**Why it exists:** Tests must not be retrieved as 'library code', and the agent must never edit a test to make it pass (that would be cheating).

In [4]:
from ast_agent.indexer import is_test_path
for p in ["tests/test_minilib.py", "minilib/stats.py", "conftest.py", "pkg/test_utils.py"]:
    print(f"{p:25s} -> {is_test_path(p)}")

tests/test_minilib.py     -> True
minilib/stats.py          -> False
conftest.py               -> True
pkg/test_utils.py         -> True


### `Symbol (dataclass)`

```python
@dataclass
class Symbol:
    id, path, name, kind, start, end, code, docstring, calls, synthetic
```

**What it is:** A *dataclass* is a class that mainly stores data. `Symbol` is one indexed piece of code. Its fields: `id` (unique key like `"minilib/stats.py::median"`), `path` (file), `name` (`"Rectangle.area"` for methods), `kind` (`function`, `method`, `class` or `module`), `start`/`end` (first and last line, 1-based), `code` (the source text shown to the model), `docstring`, `calls` (names of functions it calls) and `synthetic` (`True` for summaries that are not one contiguous block of lines, i.e. class outlines and module headers).

**Returns:** An object holding those fields.

**Why it exists:** Every other part of the system talks about 'symbols': the retriever ranks them, the agent shows them, and the evaluation checks whether the buggy one was shown.

### `Symbol.short_name`

```python
@property
def short_name(self) -> str
```

**What it is:** The last part of a dotted name: `"Rectangle.area"` becomes `"area"`. A `@property` is used like an attribute (`sym.short_name`, no brackets).

**Returns:** A string.

**Why it exists:** Function calls in code use the short name (`self.area()` or `area()`), so the call graph matches symbols by short name.

### `_called_names`

```python
def _called_names(node: ast.AST) -> set[str]
```

**What it is:** Walks every node inside a function's syntax tree and collects the names of everything it calls. For `foo(x)` it records `foo`; for `obj.bar()` it records `bar`.

**Inputs:**
- `node`: an AST node, usually a function definition.

**Returns:** A set of names, for example `{"_sorted_copy", "len", "ValueError"}`.

**Why it exists:** This is how the call graph is built: if `median` calls `_sorted_copy`, they are connected.

In [5]:
import ast
from ast_agent.indexer import _called_names
src = "def median(values):\n    data = _sorted_copy(values)\n    n = len(data)\n    return data.pop()\n"
print(_called_names(ast.parse(src).body[0]))

{'len', 'pop', '_sorted_copy'}


### `_start_line`

```python
def _start_line(node: ast.AST) -> int
```

**What it is:** Returns the first line of a function or class, including any decorators above it (like `@property`).

**Inputs:**
- `node`: an AST node.

**Returns:** A 1-based line number.

**Why it exists:** Python reports a function's `lineno` at the `def` line, so without this the decorator would be cut off when we copy the code.

### `_signature_lines`

```python
def _signature_lines(node: ast.AST, lines: list[str]) -> str
```

**What it is:** Returns only the header of a function: its decorators and the `def name(args):` line(s), handling signatures that span several lines.

**Inputs:**
- `node`: a function node.
- `lines`: the file's lines.

**Returns:** The header text.

**Why it exists:** Class outlines list each method by its header only, so the model sees the shape of a class without every method body.

In [6]:
from ast_agent.indexer import _start_line, _signature_lines
src = "class A:\n    @property\n    def area(self,\n             unit='m'):\n        return 1\n"
lines = src.splitlines()
method = ast.parse(src).body[0].body[0]
print("def line:", method.lineno, "| start incl. decorator:", _start_line(method))
print(_signature_lines(method, lines))

def line: 3 | start incl. decorator: 2
    @property
    def area(self,
             unit='m'):


### `_Visitor` (class) and its methods

**What it is:** an `ast.NodeVisitor`, Python's standard tool for walking a syntax tree. When it meets a node of type `X`, it calls the method `visit_X`. Our visitor turns functions and classes into `Symbol` objects.

| Method | Definition | Why |
|---|---|---|
| `__init__(rel_path, lines)` | Stores the file path and its lines, starts an empty `symbols` list and an empty `class_stack`. | `class_stack` remembers which class we are inside, to build names like `Rectangle.area`. |
| `_segment(start, end)` | Returns lines `start`..`end` of the file as one string. | Copies the exact source of a symbol. |
| `_add_function(node)` | Creates a `function` or `method` symbol (method if we are inside a class) with its code, docstring and called names. Nested functions are **not** indexed separately. | A helper inside a function belongs with its parent, so they are shown together. |
| `visit_FunctionDef(node)` | Called for each `def`; calls `_add_function`. | Normal functions. |
| `visit_AsyncFunctionDef(node)` | Called for each `async def`; calls `_add_function`. | Async functions are indexed the same way. |
| `visit_ClassDef(node)` | Creates a **class outline** symbol (header, class-level lines, method signatures with `...`), then visits the methods inside it with the class name pushed on `class_stack`. | The outline gives the model the "map" of a class in very few tokens; each method is still its own symbol. |

### `RepoIndex (class)`

```python
class RepoIndex:
    def __init__(self, root, symbols, files)
```

**What it is:** The finished index of a repository. It holds `root` (absolute folder), `symbols` (all `Symbol`s), `files` (path → full text), and two lookup dictionaries: `by_id` (symbol id → symbol) and `by_short_name` (short name → list of symbols with that name).

**Inputs:**
- `root`: repository folder.
- `symbols`: list of symbols.
- `files`: dict of file texts.

**Returns:** A `RepoIndex` object.

**Why it exists:** The lookup dictionaries make the call-graph and gold-symbol checks fast.

### `RepoIndex.build`

```python
@classmethod
def build(cls, root: str, include_tests: bool = False) -> RepoIndex
```

**What it is:** The normal way to create an index. It walks every folder (skipping `SKIP_DIRS` and hidden folders), reads every `.py` file that is not a test and not larger than 1 MB, and parses it into symbols with `_parse`. A `@classmethod` is called on the class itself: `RepoIndex.build(path)`.

**Inputs:**
- `root`: repository folder.
- `include_tests`: set `True` to index test files too (not used in the paper).

**Returns:** A `RepoIndex`.

**Why it exists:** One call turns a whole repository into searchable pieces.

### `RepoIndex._parse`

```python
@staticmethod
def _parse(rel: str, text: str) -> list[Symbol]
```

**What it is:** Parses one file. It runs `_Visitor` over the top-level functions and classes, then adds one **module header** symbol containing everything else at the top level (imports, constants). Files with syntax errors give an empty list instead of crashing.

**Inputs:**
- `rel`: the file's relative path.
- `text`: the file's source.

**Returns:** The file's symbols.

**Why it exists:** The module header lets the model see imports and constants such as `DEFAULT_PRECISION` without reading the whole file.

In [7]:
from ast_agent.indexer import RepoIndex
index = RepoIndex.build(SAMPLE_REPO)
print(f"{len(index.files)} files, {len(index.symbols)} symbols\n")
for s in index.symbols:
    print(f"{s.kind:8s} {s.id:42s} lines {s.start}-{s.end}  synthetic={s.synthetic}")

4 files, 21 symbols

module   minilib/__init__.py::<module>              lines 1-1  synthetic=True
class    minilib/shapes.py::Rectangle               lines 4-30  synthetic=True
method   minilib/shapes.py::Rectangle.__init__      lines 7-11  synthetic=False
method   minilib/shapes.py::Rectangle.area          lines 13-14  synthetic=False
method   minilib/shapes.py::Rectangle.perimeter     lines 16-17  synthetic=False
method   minilib/shapes.py::Rectangle.contains      lines 19-23  synthetic=False
method   minilib/shapes.py::Rectangle.overlaps      lines 25-30  synthetic=False
function minilib/shapes.py::total_area              lines 33-34  synthetic=False
module   minilib/shapes.py::<module>                lines 1-34  synthetic=True
function minilib/stats.py::mean                     lines 6-10  synthetic=False
function minilib/stats.py::_sorted_copy             lines 13-14  synthetic=False
function minilib/stats.py::median                   lines 17-26  synthetic=False
function minilib

Here is what a **class outline** looks like: method bodies are replaced by `...`, so the whole class costs only a few tokens.

In [8]:
print(index.by_id["minilib/shapes.py::Rectangle"].code)

class Rectangle:
    """Rectangle with its lower-left corner at (x, y)."""
    def __init__(self, x, y, width, height): ...
    def area(self): ...
    def perimeter(self): ...
    def contains(self, px, py): ...
    def overlaps(self, other): ...


### `RepoIndex.callees`

```python
def callees(self, sym: Symbol) -> list[Symbol]
```

**What it is:** Finds the functions and methods that `sym` calls, by looking up each name in `sym.calls` in `by_short_name`.

**Inputs:**
- `sym`: a symbol.

**Returns:** A list of symbols that `sym` calls.

**Why it exists:** If the bug report mentions `median`, the bug might really be in a helper that `median` calls.

### `RepoIndex.callers`

```python
def callers(self, sym: Symbol) -> list[Symbol]
```

**What it is:** The reverse: every function or method whose `calls` contains `sym`'s short name.

**Inputs:**
- `sym`: a function or method symbol.

**Returns:** A list of symbols that call `sym`.

**Why it exists:** If a test fails in `variance`, the cause might be in `mean`, which `variance` calls, and vice versa.

### `RepoIndex.symbol_at`

```python
def symbol_at(self, rel_path: str, line: int) -> Symbol | None
```

**What it is:** Finds the smallest real (non-synthetic) symbol containing a given line of a file.

**Inputs:**
- `rel_path`: file path.
- `line`: line number.

**Returns:** The symbol, or `None` if the line is outside every function.

**Why it exists:** Tracebacks report *file + line*; this converts them into *which function*. The bug generator also uses it to know which function a mutation is in.

In [9]:
med = index.by_id["minilib/stats.py::median"]
var = index.by_id["minilib/stats.py::variance"]
mean_ = index.by_id["minilib/stats.py::mean"]
print("median calls:      ", [s.name for s in index.callees(med)])
print("variance calls:    ", [s.name for s in index.callees(var)])
print("who calls mean():  ", [s.name for s in index.callers(mean_)])
print("line", med.start + 3, "of stats.py is inside:", index.symbol_at("minilib/stats.py", med.start + 3).name)

median calls:       ['_sorted_copy']
variance calls:     ['mean']
who calls mean():   ['variance', 'rounded_mean']
line 20 of stats.py is inside: median


## 3. `retrieval.py`: choosing what the model sees

**Goal:** given the bug report (the *query*) and a token budget, return the pieces of code most likely to contain the bug. The paper compares four strategies:

| Name | Strategy |
|---|---|
| `file` | Rank **whole files** with BM25 and pack them until the budget is full (the last file gets cut). This is the baseline. |
| `ast` | Rank **symbols** with BM25. |
| `ast_graph` | `ast` plus the **callers and callees** of the top hits. This is the paper's retriever. |
| `oracle` | Always include the **gold** (buggy) function first. This is a cheating upper bound, for comparison only. |

All four use the same budget, so any difference comes from *what* is shown, not *how much*.

### `estimate_tokens`

```python
def estimate_tokens(text: str) -> int
```

**What it is:** A rough token count: number of characters divided by 4 (at least 1).

**Inputs:**
- `text`: any text.

**Returns:** An integer.

**Why it exists:** Every retriever needs to measure size the same way. Counting with a real tokenizer would tie the code to one model; the same rough estimate is used for every condition, so the comparison stays fair.

**`_STOP`** is a set of *stop words*: very common words like `the`, `is`, `self`, `return`, `def` that appear everywhere and tell us nothing about where a bug is, so the tokenizer drops them.

### `tokenize`

```python
def tokenize(text: str) -> list[str]
```

**What it is:** Splits text into lowercase search words. It is *identifier-aware*: `parse_date` produces `parse_date`, `parse` and `date`, and `camelCase` is split too. Stop words and one-letter words are removed.

**Inputs:**
- `text`: a bug report, a file, or a symbol's code.

**Returns:** A list of words.

**Why it exists:** Bug reports mention `test_median_odd_even`; the code contains `median`. Splitting identifiers lets these match.

In [10]:
from ast_agent.retrieval import estimate_tokens, tokenize
print(tokenize("tests/test_minilib.py::test_median_odd_even - assert 1.67 == 2.5"))
print(tokenize("def parseDate(self, raw_value): return None"))
print("tokens in stats.py ≈", estimate_tokens(index.files["minilib/stats.py"]))

['tests', 'minilib', 'test_minilib', 'test', 'odd', 'test', 'median', 'even', 'test_median_odd_even', 'assert']
['parse', 'date', 'parsedate', 'value', 'raw', 'raw_value']
tokens in stats.py ≈ 273


### `BM25 (class)`

```python
class BM25:
    def __init__(self, docs: list[list[str]], k1=1.5, b=0.75)
    def scores(self, query: list[str]) -> list[float]
```

**What it is:** The BM25 ranking formula used by search engines. `__init__` pre-computes, for each document, how often each word occurs (`tfs`), the document lengths, the average length, and each word's *IDF* (inverse document frequency: rare words get high weight). `scores` returns one relevance score per document for a query.

**Inputs:**
- `docs`: list of documents, each a list of words (from `tokenize`).
- `k1`: how quickly repeated words stop adding score (1.5).
- `b`: how much long documents are penalised (0.75).
- `query`: list of query words.

**Returns:** `scores` returns a list of floats, one per document; higher means more relevant.

**Why it exists:** BM25 is simple, fast, needs no GPU and no training, and is a strong standard baseline [Robertson & Zaragoza, 2009].

**The formula, in words.** For each query word that appears in a document: `score += IDF(word) × f×(k1+1) / (f + k1×(1 − b + b×length/avg_length))`, where `f` is how many times the word occurs in that document. The more often a rare word appears, the higher the score, with diminishing returns, and long documents are slightly penalised.

In [11]:
from ast_agent.retrieval import BM25
docs = [tokenize("def mean(values): return sum(values) / len(values)"),
        tokenize("def median(values): data = sorted(values); mid = n // 2"),
        tokenize("def slugify(title): return title.lower()")]
bm = BM25(docs)
for name, s in zip(["mean", "median", "slugify"], bm.scores(tokenize("median returns wrong middle value"))):
    print(f"{name:8s} {s:.3f}")

mean     0.000
median   0.929
slugify  0.000


### `ContextChunk (dataclass)`

```python
@dataclass
class ContextChunk:
    path, code, label, start=0, end=0, symbol_id=None, score=0.0
```

**What it is:** One piece of code placed in the prompt: which file, the code, a label like `"function median"` or `"file (truncated)"`, its line range (0 for outlines/headers), which symbol it came from, and its retrieval score.

**Returns:** A `ContextChunk`.

**Why it exists:** All retrievers return the same kind of object, so the rest of the agent does not care which retriever was used.

### `ContextChunk.tokens`

```python
@property
def tokens(self) -> int
```

**What it is:** The estimated token size of the chunk's code (`estimate_tokens(self.code)`).

**Returns:** An integer.

**Why it exists:** Used to stay inside the budget.

### `ContextChunk.covers`

```python
def covers(self, path: str, start: int, end: int) -> bool
```

**What it is:** Checks whether this chunk contains the whole line range `start`..`end` of `path`.

**Inputs:**
- `path, start, end`: a file and line range, e.g. the gold function's.

**Returns:** `True` or `False`.

**Why it exists:** This is how *symbol hit* is measured for whole-file retrieval: was the complete buggy function inside the shown text, or was it cut off?

**`_TB_PATTERNS`** are two regular expressions that recognise traceback lines in both formats Python and pytest print: `File "path.py", line 12` and `path.py:12:`.

### `traceback_frames`

```python
def traceback_frames(text: str) -> list[tuple[str, int]]
```

**What it is:** Finds every *(file, line)* pair mentioned in an error message.

**Inputs:**
- `text`: test output or a bug report.

**Returns:** A list like `[("minilib/stats.py", 23)]`.

**Why it exists:** The first step in turning an error message into code locations.

### `traceback_symbols`

```python
def traceback_symbols(index: RepoIndex, text: str) -> list[Symbol]
```

**What it is:** Converts each traceback frame into the symbol at that line (using `RepoIndex.symbol_at`), after stripping the repository's absolute path. Frames outside the repository (Python's own files, pytest) are ignored, and duplicates are removed.

**Inputs:**
- `index`: the repo index.
- `text`: test output.

**Returns:** A list of symbols, in traceback order.

**Why it exists:** Used twice: to boost symbols named in a traceback during ranking, and in self-correction to show the model code from the failing traceback that it has not seen yet (*traceback retrieval*).

In [12]:
from ast_agent.retrieval import traceback_frames, traceback_symbols
fake_output = f'File "{index.root}/minilib/stats.py", line {med.start + 8}, in median\nminilib/stats.py:{var.start + 2}: in variance'
print(traceback_frames(fake_output))
print([s.id for s in traceback_symbols(index, fake_output)])

[('gemma-ast-agent/benchmarks/sample/repo/minilib/stats.py', 25), ('minilib/stats.py', 31)]
['minilib/stats.py::median', 'minilib/stats.py::variance']


### `Retriever (base class)`

```python
class Retriever:
    def __init__(self, index)
    def retrieve(self, query, budget) -> list[ContextChunk]
```

**What it is:** A template that every retriever follows: it holds the index and must provide `retrieve(query, budget)`. The base version just raises `NotImplementedError`.

**Inputs:**
- `query`: the bug report.
- `budget`: max tokens of code.

**Returns:** A list of `ContextChunk`s.

**Why it exists:** Swapping retrievers is then a one-word change, which is exactly what an ablation study needs.

### `FileRetriever.retrieve`

```python
def retrieve(self, query: str, budget: int) -> list[ContextChunk]
```

**What it is:** The **baseline**. Scores every whole file with BM25 (path + contents), then adds files from best to worst until the budget is used. The file that does not fit is **truncated** line by line.

**Inputs:**
- `query`: bug report.
- `budget`: token budget.

**Returns:** Whole-file chunks (the last may be marked `file (truncated)`).

**Why it exists:** This is what a simple agent would do. On more-itertools the main file is 5,644 lines, so truncation usually cut the buggy function off: symbol hit was only 4%.

### `ASTRetriever.__init__`

```python
def __init__(self, index, graph=False, graph_seeds=3, graph_decay=0.5)
```

**What it is:** Builds a BM25 index over **symbols**. Each symbol's searchable text is its name (repeated 3 times, so names count most), its path, docstring and code.

**Inputs:**
- `graph`: `True` = also add call-graph neighbours (`ast_graph`).
- `graph_seeds`: how many top hits get neighbours (3).
- `graph_decay`: neighbours get this fraction of their seed's score (0.5).

**Returns:** An `ASTRetriever`.

**Why it exists:** Ranking functions instead of files gives much more precise context.

### `ASTRetriever.rank`

```python
def rank(self, query: str) -> list[tuple[float, Symbol]]
```

**What it is:** Scores all symbols with BM25, then adds two bonuses: **+0.5 × top score** if the bug report names the symbol, and **+1.0 × top score** if the symbol appears in a traceback inside the report. Returns symbols with a positive score, best first.

**Inputs:**
- `query`: bug report.

**Returns:** A sorted list of `(score, symbol)` pairs.

**Why it exists:** Exact names and traceback frames are the strongest clues a bug report can contain, stronger than word overlap.

### `ASTRetriever.retrieve`

```python
def retrieve(self, query: str, budget: int) -> list[ContextChunk]
```

**What it is:** Ranks symbols; if `graph=True`, adds the callers and callees of the top 3 symbols at half their score (keeping the higher score if a symbol was already ranked); then packs them into the budget with `pack_symbols`.

**Inputs:**
- `query`: bug report.
- `budget`: token budget.

**Returns:** Symbol chunks in file order.

**Why it exists:** Call-graph expansion catches bugs that sit one step away from the function the report mentions.

### `symbol_chunk`

```python
def symbol_chunk(sym: Symbol, score: float = 0.0) -> ContextChunk
```

**What it is:** Wraps a `Symbol` as a `ContextChunk` with a readable label (`function median`, `class outline Rectangle`, `module header <module>`). Synthetic symbols get line range 0.

**Inputs:**
- `sym`: a symbol.
- `score`: its retrieval score.

**Returns:** A `ContextChunk`.

**Why it exists:** Converts index objects into prompt objects.

### `pack_symbols`

```python
def pack_symbols(ranked, budget, already=None) -> list[ContextChunk]
```

**What it is:** A *greedy knapsack*: goes through symbols from best to worst score and adds each one that still fits in the budget. If one is too big, it skips it and keeps trying smaller ones. Finally it sorts the chosen chunks into file order.

**Inputs:**
- `ranked`: list of `(score, symbol)`.
- `budget`: token budget.
- `already`: ids to skip (already shown).

**Returns:** A list of `ContextChunk`s.

**Why it exists:** Uses the budget well, and file order lets the model read code naturally, top to bottom.

### `OracleRetriever`

```python
class OracleRetriever(ASTRetriever):
    def __init__(self, index, gold_ids)
    def retrieve(self, query, budget)
```

**What it is:** Gives the gold (buggy) symbols an enormous score so they are always included first, then fills the rest of the budget with the normal ranking.

**Inputs:**
- `gold_ids`: the ids of the buggy functions.

**Returns:** Chunks that always contain the bug.

**Why it exists:** An *upper bound on localization*. If even the oracle fails, the problem is not finding the bug but fixing it. In the paper the oracle reached only 34% on boltons, which showed that retrieval alone was not the bottleneck.

### `make_retriever`

```python
def make_retriever(name: str, index: RepoIndex, gold_ids=None) -> Retriever
```

**What it is:** A small *factory*: turns a name (`"file"`, `"ast"`, `"ast_graph"`, `"oracle"`) into the matching retriever object.

**Inputs:**
- `name`: retriever name.
- `index`: repo index.
- `gold_ids`: needed only for `oracle`.

**Returns:** A retriever; raises `ValueError` for an unknown name.

**Why it exists:** Lets experiment settings be plain strings.

**Demo:** the same bug report with all four retrievers and a deliberately tiny budget of 120 tokens, so they have to choose.

In [13]:
from ast_agent.retrieval import make_retriever
issue = "The following tests fail:\n- tests/test_minilib.py::test_median_odd_even - assert 1.67 == 2.5"
for name in ["file", "ast", "ast_graph", "oracle"]:
    r = make_retriever(name, index, gold_ids=["minilib/stats.py::median"])
    chunks = r.retrieve(issue, budget=120)
    print(f"{name:10s}", [c.symbol_id or f"{c.path} ({c.label})" for c in chunks])

file       ['minilib/stats.py (file (truncated))']
ast        ['minilib/__init__.py::<module>', 'minilib/shapes.py::Rectangle.area', 'minilib/stats.py::median', 'minilib/text.py::<module>']
ast_graph  ['minilib/__init__.py::<module>', 'minilib/stats.py::_sorted_copy', 'minilib/stats.py::median', 'minilib/text.py::<module>']
oracle     ['minilib/__init__.py::<module>', 'minilib/shapes.py::Rectangle.area', 'minilib/stats.py::median', 'minilib/text.py::<module>']


## 4. `prompts.py`: what we actually say to the model

All prompt text lives in one file, so the paper can quote it exactly and every experiment uses identical wording.

**`INSTRUCTIONS`** is the fixed instruction block: explain the root cause in at most 3 sentences, then give edit blocks in the SEARCH/REPLACE format; SEARCH must match exactly and only once; tests are read-only; make the smallest fix.

In [14]:
from ast_agent import prompts
print(prompts.INSTRUCTIONS)

You are a software engineering agent. Fix the issue below by editing the repository.

Reply with a short root-cause explanation (at most 3 sentences), then one or more edit blocks in exactly this format:

path/to/file.py
<<<<<<< SEARCH
exact lines copied from the current code
replacement lines
>>>>>>> REPLACE

Rules:
- SEARCH must match the current file exactly (including indentation) and occur only once; include 1-3 surrounding lines if needed.
- Edit library code only. Test files are read-only.
- Make the smallest change that fixes the root cause.


### `format_chunks`

```python
def format_chunks(chunks: list[ContextChunk]) -> str
```

**What it is:** Renders retrieved chunks as Markdown: a heading with file, line range and label, then the code in a ```python block. Prints `(no code retrieved)` if the list is empty.

**Inputs:**
- `chunks`: retrieved code.

**Returns:** A string.

**Why it exists:** Clear headings with line numbers help the model copy SEARCH lines exactly.

### `format_file_list`

```python
def format_file_list(paths: list[str], limit: int = 80) -> str
```

**What it is:** Lists repository file paths, at most 80, with a note like `... and 12 more`.

**Inputs:**
- `paths`: all file paths.
- `limit`: how many to show.

**Returns:** A string.

**Why it exists:** Gives the model a map of the repository at very low cost.

### `initial_prompt`

```python
def initial_prompt(issue, chunks, file_paths) -> str
```

**What it is:** Builds the first message: instructions + `## Issue` + `## Repository files` + `## Relevant code`.

**Inputs:**
- `issue`: bug report.
- `chunks`: retrieved code.
- `file_paths`: all files.

**Returns:** The full prompt text.

**Why it exists:** This single message is everything the model knows on attempt 1.

In [15]:
chunks = make_retriever("ast_graph", index).retrieve(issue, 300)
print(prompts.initial_prompt(issue, chunks, sorted(index.files))[-900:])

es):
    """Population variance."""
    m = mean(values)
    return sum((x - m) ** 2 for x in values) / len(values)
```

### minilib/stats.py (lines 46-47) - function rounded_mean
```python
def rounded_mean(values, precision=DEFAULT_PRECISION):
    return round(mean(values), precision)
```

### minilib/stats.py (outline, not contiguous) - module header <module>
```python
"""Small statistics helpers."""
DEFAULT_PRECISION = 2
```

### minilib/text.py (lines 11-12) - function word_count
```python
def word_count(text):
    return len([w for w in text.split() if w])
```

### minilib/text.py (lines 25-27) - function is_palindrome
```python
def is_palindrome(text):
    cleaned = [c for c in text.lower() if c.isalnum()]
    return cleaned == cleaned[::-1] and len(cleaned) > 0
```

### minilib/text.py (outline, not contiguous) - module header <module>
```python
"""Text utilities."""
import re
```


### `no_edits_feedback`

```python
def no_edits_feedback() -> str
```

**What it is:** The message sent when the model's reply contained no edit blocks at all: it reminds the model of the exact format.

**Returns:** A string.

**Why it exists:** Small models sometimes answer in prose; this nudges them back to the format.

### `apply_error_feedback`

```python
def apply_error_feedback(errors: list[str]) -> str
```

**What it is:** The message sent when edits could not be applied (for example SEARCH text not found). It lists each error and asks for corrected blocks.

**Inputs:**
- `errors`: error messages from `apply_edits`.

**Returns:** A string.

**Why it exists:** Specific error messages let the model fix its formatting mistake.

### `test_failure_feedback`

```python
def test_failure_feedback(diff, test_output, extra) -> str
```

**What it is:** The **self-correction** message: the previous patch (as a diff), the tail of the failing test output, and optionally code from the traceback the model had not seen. It says the repository has been reset and asks for a complete new patch.

**Inputs:**
- `diff`: the failed patch.
- `test_output`: failing test output.
- `extra`: extra chunks from traceback retrieval.

**Returns:** A string.

**Why it exists:** This is the paper's key ingredient: concrete feedback such as `assert 1.67 == 2.5` tells a small model far more than extra retrieved code.

In [16]:
print(prompts.test_failure_feedback("-    x = 1\n+    x = 2", "FAILED test_x - assert 2 == 1", [])[:400])

Your patch applied cleanly but the tests still fail.

## Your previous patch
```diff
-    x = 1
+    x = 2
```

## Test output (tail)
```
FAILED test_x - assert 2 == 1
```

The repository has been reset to its ORIGINAL state. Reply with a complete, corrected set of edit blocks written against the original code.


## 5. `models.py`: talking to Gemma (or a fake model)

Every model backend has the same small interface: `generate(messages, max_tokens, temperature)` returns a `Generation`. *Messages* are a chat history: a list of `{"role": "user" or "assistant", "content": text}`. No `system` role is used, because some Gemma chat templates reject it.

### `Generation (dataclass)`

```python
@dataclass
class Generation:
    text, prompt_tokens, completion_tokens, latency_s
```

**What it is:** One model reply: the text, how many tokens were read (prompt) and written (completion), and how many seconds it took.

**Returns:** A `Generation`.

**Why it exists:** The agent adds these numbers up to report cost (tokens and model time per task).

### `ModelClient (Protocol)`

```python
class ModelClient(Protocol):
    name: str
    def generate(self, messages, max_tokens=2048, temperature=0.0) -> Generation
```

**What it is:** A `Protocol` is Python's way of saying *any object with these methods counts*. Nothing inherits from it; it documents the interface.

**Returns:** —

**Why it exists:** The agent works with any backend that has `generate`, whether that is a GPU model, a server, or a fake.

### `_prompt_estimate`

```python
def _prompt_estimate(messages: list[dict]) -> int
```

**What it is:** Adds up `estimate_tokens` over all messages.

**Inputs:**
- `messages`: chat history.

**Returns:** An integer.

**Why it exists:** Fallback token count when a backend does not report real usage.

### `OpenAICompatibleClient`

```python
class OpenAICompatibleClient:
    def __init__(self, base_url, model, api_key='none', timeout=900, extra_body=None)
    def generate(self, messages, max_tokens=2048, temperature=0.0) -> Generation
```

**What it is:** Sends the chat to any server that speaks the OpenAI `/v1/chat/completions` format (vLLM, Ollama, LM Studio, llama.cpp). It uses only Python's built-in `urllib`, so no extra packages are needed. It reads the reply text and token usage from the JSON response.

**Inputs:**
- `base_url`: e.g. `http://localhost:11434/v1` for Ollama.
- `model`: model name on that server.
- `api_key`: usually unused locally.
- `timeout`: seconds to wait.
- `extra_body`: extra JSON fields for the server.

**Returns:** A `Generation`.

**Why it exists:** Lets you run the agent against a local Gemma 4 on your own laptop.

### `TransformersClient`

```python
class TransformersClient:
    def __init__(self, model_id, dtype='auto', device_map='auto', load_in_4bit=False)
    def generate(self, messages, max_tokens=2048, temperature=0.0) -> Generation
```

**What it is:** Loads a Hugging Face model in the same Python process (this is what the Kaggle runs used). `generate` applies Gemma's chat template with `enable_thinking=False` (direct answers), runs greedy decoding when `temperature == 0` (always the most likely next token, so results are reproducible), and decodes only the new tokens.

**Inputs:**
- `model_id`: model path or Hugging Face id.
- `dtype`: `float16` on T4 GPUs.
- `device_map`: `auto` spreads the model over available GPUs.
- `load_in_4bit`: compress the model to fit smaller GPUs.

**Returns:** A `Generation`.

**Why it exists:** Runs Gemma 4 on Kaggle's free 2× T4 GPUs. `torch` and `transformers` are imported only here, so the rest of the package has no heavy dependencies.

### `ScriptedClient`

```python
class ScriptedClient:
    def __init__(self, responses, name='scripted')
    def generate(self, messages, ...) -> Generation
```

**What it is:** A **fake model**. It returns the next answer from a list (repeating the last one when the list runs out), or calls a function you provide on the message history. `calls` counts how many times it was asked.

**Inputs:**
- `responses`: a list of reply strings, or a function `messages -> reply`.

**Returns:** A `Generation`.

**Why it exists:** Lets us test the whole agent, including self-correction, in seconds without a GPU. This notebook uses it everywhere.

### `GoldPatchClient`

```python
class GoldPatchClient(ScriptedClient):
    def __init__(self, bug: dict)
```

**What it is:** A fake model that always answers with the **known correct fix** for a planted bug: SEARCH = the buggy line(s), REPLACE = the original line(s).

**Inputs:**
- `bug`: a task's bug record (`path`, `buggy_snippet`, `original_snippet`).

**Returns:** A client.

**Why it exists:** A **harness sanity check**: if the perfect answer does not resolve a task, the task or the harness is broken, not the model. Every benchmark task passed this check.

In [17]:
from ast_agent.models import ScriptedClient, GoldPatchClient
fake = ScriptedClient(["first answer", "second answer"])
for _ in range(3):
    print(fake.generate([{"role": "user", "content": "hi"}]).text)
print("calls:", fake.calls)

tasks = [json.loads(l) for l in open(os.path.join(SAMPLE, "tasks.jsonl"))]
task = next(t for t in tasks if t["id"] == "minilib-002")
print("\nGold answer for minilib-002:\n" + GoldPatchClient(task["bug"]).generate([]).text)

first answer
second answer
second answer
calls: 3

Gold answer for minilib-002:
Reverting the planted mutation.

minilib/stats.py
<<<<<<< SEARCH
    return (data[mid - 1] + data[mid]) / 3
    return (data[mid - 1] + data[mid]) / 2
>>>>>>> REPLACE



## 6. `edits.py`: turning the model's answer into a code change

The model writes edits like this:

```
minilib/stats.py
<<<<<<< SEARCH
    return (data[mid - 1] + data[mid]) / 3
=======
    return (data[mid - 1] + data[mid]) / 2
>>>>>>> REPLACE
```

This module finds those blocks in the reply and applies them **safely and atomically**: either every block applies, or nothing is written. The regular expressions `_SEARCH`, `_DIVIDER`, `_REPLACE` and `_FENCE` recognise the marker lines (`<<<<<<< SEARCH`, `=======`, `>>>>>>> REPLACE`, and ``` fences), allowing 5 to 9 marker characters because models are not always exact.

### `Edit (dataclass)`

```python
@dataclass
class Edit:
    path, search, replace
```

**What it is:** One requested change: which file, the exact old text, and the new text.

**Returns:** An `Edit`.

**Why it exists:** The parsed, structured version of one block.

### `ApplyResult (dataclass)`

```python
@dataclass
class ApplyResult:
    ok, errors, changed
```

**What it is:** The outcome of applying edits: `ok` (success?), `errors` (messages for the model), `changed` (file → (old text, new text)).

**Returns:** An `ApplyResult`.

**Why it exists:** Keeps the old text so the change can be undone and shown as a diff.

### `ApplyResult.diff`

```python
def diff(self) -> str
```

**What it is:** Produces a *unified diff* (lines starting with `-` removed, `+` added) for every changed file, using Python's `difflib`.

**Returns:** A diff string.

**Why it exists:** The diff is the 'patch' saved in the results and shown back to the model during self-correction.

### `_clean_path`

```python
def _clean_path(line: str) -> str
```

**What it is:** Cleans the file-path line written above a block: strips backticks, asterisks, `#`, colons, quotes and prefixes like `File:` or `path:`.

**Inputs:**
- `line`: the line before `<<<<<<< SEARCH`.

**Returns:** A clean relative path.

**Why it exists:** Models decorate paths in many ways (`**minilib/stats.py**`); this accepts them all.

### `parse_edits`

```python
def parse_edits(text: str) -> list[Edit]
```

**What it is:** Scans the model's reply line by line. For each `<<<<<<< SEARCH` it takes the nearest non-empty line above it as the path, collects lines until `=======` as SEARCH, then lines until `>>>>>>> REPLACE` as REPLACE. Anything before a `</think>` tag is ignored.

**Inputs:**
- `text`: the model's full reply.

**Returns:** A list of `Edit`s (empty if none were found).

**Why it exists:** Turns free text into structured changes.

> **Jupyter gotcha:** inside a notebook cell, IPython silently rewrites any line that *starts with* `>>>` (it thinks you pasted a Python prompt), so typing `>>>>>>> REPLACE` directly in a multi-line string breaks it. The demos therefore build the marker lines in code with `">" * 7`. This only affects typing the markers into notebook cells, not the agent itself.

In [18]:
from ast_agent.edits import parse_edits, _clean_path

# Build the three marker lines in code (see the note above).
SEARCH, DIVIDER, REPLACE = "<" * 7 + " SEARCH", "=" * 7, ">" * 7 + " REPLACE"
def edit_block(path, old, new):
    return f"{path}\n{SEARCH}\n{old}\n{DIVIDER}\n{new}\n{REPLACE}\n"

reply = ("The median divides by 3 instead of 2.\n\n" +
         edit_block("**minilib/stats.py**",
                    "    return (data[mid - 1] + data[mid]) / 3",
                    "    return (data[mid - 1] + data[mid]) / 2"))
print(reply)
print("cleaned path:", _clean_path("**File: `minilib/stats.py`**"))
for e in parse_edits(reply):
    print(e)

The median divides by 3 instead of 2.

**minilib/stats.py**
<<<<<<< SEARCH
    return (data[mid - 1] + data[mid]) / 3
    return (data[mid - 1] + data[mid]) / 2
>>>>>>> REPLACE

cleaned path: minilib/stats.py
Edit(path='minilib/stats.py', search='    return (data[mid - 1] + data[mid]) / 3', replace='    return (data[mid - 1] + data[mid]) / 2')


### `_find_unique`

```python
def _find_unique(haystack: str, needle: str) -> tuple[int, str | None]
```

**What it is:** Looks for `needle` in `haystack` and requires exactly **one** match.

**Inputs:**
- `haystack`: file content.
- `needle`: SEARCH text.

**Returns:** `(position, None)` if found once; `(-1, "not found")` or `(-1, "matches N places...")` otherwise.

**Why it exists:** If the SEARCH text matched twice we would not know which place to change, so we refuse and tell the model to add context.

### `_indent`

```python
def _indent(s: str) -> str
```

**What it is:** Returns the leading spaces/tabs of a line.

**Inputs:**
- `s`: a line.

**Returns:** The indentation string.

**Why it exists:** Used to re-indent replacements.

### `_fuzzy_replace`

```python
def _fuzzy_replace(content, search, replace) -> tuple[str | None, str | None]
```

**What it is:** A fallback when the exact match fails: compares lines **ignoring indentation and trailing spaces**. If exactly one place matches, it shifts the REPLACE lines to the file's real indentation and substitutes them.

**Inputs:**
- `content`: file text.
- `search`: SEARCH text.
- `replace`: REPLACE text.

**Returns:** `(new_content, None)` on success, or `(None, error)`.

**Why it exists:** Small models often get indentation slightly wrong; rejecting those otherwise-correct patches would unfairly hurt them.

In [19]:
from ast_agent.edits import _find_unique, _fuzzy_replace, _indent
content = "def f():\n    x = 1\n    return x\n"
print(_find_unique(content, "x = 1"), _find_unique(content, "x"), _find_unique(content, "y = 2"))
print(repr(_indent("        return 3")))
new, err = _fuzzy_replace(content, "x = 1\nreturn x", "x = 2\nreturn x")  # no indentation given
print(new)

(13, None) (-1, 'matches 2 places; add more surrounding lines') (-1, 'not found')
'        '
def f():
    x = 2
    return x



### `_is_protected`

```python
def _is_protected(rel: str, protected: tuple[str, ...]) -> bool
```

**What it is:** Checks whether a path matches any protected pattern (by default `TEST_GLOBS`).

**Inputs:**
- `rel`: file path.
- `protected`: patterns.

**Returns:** `True` if editing is forbidden.

**Why it exists:** The agent must fix the library, not weaken the tests.

### `apply_edits`

```python
def apply_edits(root, edits, protected=TEST_GLOBS, write=True) -> ApplyResult
```

**What it is:** Applies all edits **atomically**. For each edit it: (1) rejects absolute paths or paths that escape the repository, (2) rejects test files, (3) loads the file (or the version already modified by an earlier block), (4) tries an exact unique match, then `_fuzzy_replace`. Then it checks that every changed `.py` file still **parses** (no `SyntaxError`). Only if there are no errors at all does it write the files.

**Inputs:**
- `root`: repository folder.
- `edits`: parsed edits.
- `protected`: forbidden patterns.
- `write`: `False` = dry run.

**Returns:** An `ApplyResult`; `ok=False` with clear `errors` if anything went wrong.

**Why it exists:** Safety (never touch tests or files outside the repo) and clean feedback: a half-applied patch would leave the repo in a confusing state.

### `revert`

```python
def revert(root: str, result: ApplyResult) -> None
```

**What it is:** Undoes a successful `apply_edits`: writes back each file's old text, and deletes files that the edit newly created.

**Inputs:**
- `root`: repository folder.
- `result`: the `ApplyResult` to undo.

**Returns:** Nothing.

**Why it exists:** Self-correction resets the repository before every new attempt, so each patch is written against the original code.

In [20]:
from ast_agent.edits import apply_edits, revert
from ast_agent.mutate import apply_bug     # plants a recorded bug (explained in Section 9)
work = fresh_copy()
path = os.path.join(work, "minilib", "stats.py")
apply_bug(work, task["bug"])               # minilib-002: median divides by 3 instead of 2

res = apply_edits(work, parse_edits(reply))
print("ok:", res.ok)
print(res.diff())
revert(work, res)
print("after revert, bug is back:", "/ 3" in open(path).read())

bad = parse_edits("tests/test_minilib.py\n<<<<<<< SEARCH\nimport pytest\n=======\nx = 1\n>>>>>>> REPLACE")
print("editing a test:", apply_edits(work, bad).errors)
bad = parse_edits("minilib/stats.py\n<<<<<<< SEARCH\n    mid = n // 2\n=======\n    mid = (\n>>>>>>> REPLACE")
print("broken syntax: ", apply_edits(work, bad).errors)

ok: True
--- a/minilib/stats.py
+++ b/minilib/stats.py
@@ -23,7 +23,7 @@
     mid = n // 2
     if n % 2 == 1:
         return data[mid]
-    return (data[mid - 1] + data[mid]) / 3
+    return (data[mid - 1] + data[mid]) / 2
 
 
 def variance(values):

after revert, bug is back: True
editing a test: ['block 1 (tests/test_minilib.py): test files are read-only; fix the library code instead']
broken syntax:  ["minilib/stats.py: patch produces a SyntaxError at line 23: '(' was never closed"]


## 7. `runner.py`: running the tests safely

### `TestResult (dataclass)`

```python
@dataclass
class TestResult:
    passed, returncode, output, duration_s, timed_out=False
```

**What it is:** The outcome of one test run: did it pass (exit code 0), the exit code, the last part of the printed output, how long it took, and whether it was stopped for taking too long.

**Returns:** A `TestResult`.

**Why it exists:** The agent needs both the verdict and the text (for feedback).

### `_kill_group`

```python
def _kill_group(proc) -> None
```

**What it is:** Kills the test process **and every process it started** (its *process group*). On Windows it kills just the process.

**Inputs:**
- `proc`: a running `subprocess.Popen`.

**Returns:** Nothing.

**Why it exists:** A planted bug can create an infinite loop. Killing only the shell would leave pytest running forever, which really happened during development, and this function is the fix.

### `run_tests`

```python
def run_tests(cmd, cwd, timeout=300, tail_chars=4000) -> TestResult
```

**What it is:** Runs a test command inside the working copy. It puts `<repo>/src` and `<repo>` first on `PYTHONPATH`, so the tests import the *modified copy* rather than an installed version; replaces a leading `python` with the current interpreter; starts the process in its own group; and enforces the timeout with `_kill_group`.

**Inputs:**
- `cmd`: e.g. `python -m pytest -q --tb=short`.
- `cwd`: repository folder.
- `timeout`: seconds.
- `tail_chars`: how much output to keep.

**Returns:** A `TestResult`.

**Why it exists:** Reliable, isolated test runs are the ground truth of the whole experiment.

In [21]:
from ast_agent.runner import run_tests
print("clean copy :", run_tests(task["test_cmd"], fresh_copy(), timeout=120).passed)
work = fresh_copy()
apply_bug(work, task["bug"])
broken = run_tests(task["test_cmd"], work, timeout=120)
print("buggy copy :", broken.passed)
print(broken.output[-700:])

clean copy : True
buggy copy : False
                                                        [100%]
=================================== FAILURES ===================================
_____________________________ test_median_odd_even _____________________________
tests/test_minilib.py:14: in test_median_odd_even
    assert stats.median([4, 1, 3, 2]) == 2.5
E   assert 1.6666666666666667 == 2.5
E    +  where 1.6666666666666667 = <function median at 0x7f02d0749ee0>([4, 1, 3, 2])
E    +    where <function median at 0x7f02d0749ee0> = stats.median
=========================== short test summary info ============================
FAILED tests/test_minilib.py::test_median_odd_even - assert 1.666666666666666...
1 failed, 11 passed in 0.04s



**Traceback retrieval in action.** In the bug above the failing `assert` is inside the *test file*, which is not indexed, so its traceback adds no library code. Traceback retrieval helps most when the error happens *inside* the library. Task `minilib-003` flips `==` to `!=` in `mean`, so `mean` raises an error for normal input. `traceback_symbols` maps the error report to the functions involved, which is exactly the extra code the agent adds during self-correction:

In [22]:
task3 = next(t for t in tasks if t["id"] == "minilib-003")
work3 = fresh_copy(); apply_bug(work3, task3["bug"])
out3 = run_tests(task3["test_cmd"], work3, timeout=120).output
print([line for line in out3.splitlines() if "minilib/" in line and ".py:" in line][:4])
print("symbols:", [s.id for s in traceback_symbols(RepoIndex.build(work3), out3)])

['minilib/stats.py:9: in mean', 'minilib/stats.py:31: in variance', 'minilib/stats.py:9: in mean', 'minilib/stats.py:47: in rounded_mean']
symbols: ['minilib/stats.py::mean', 'minilib/stats.py::variance', 'minilib/stats.py::rounded_mean']


## 8. `agent.py`: the full loop

This is where everything comes together.

### `AgentConfig (dataclass)`

```python
@dataclass
class AgentConfig:
    retriever='ast_graph', context_budget=6000, max_attempts=3,
    traceback_retrieval=True, traceback_budget=1500, max_new_tokens=2048,
    temperature=0.0, test_timeout=300
```

**What it is:** All the agent's settings in one object: which retriever, how many tokens of code, how many attempts (1 = no self-correction), whether to add traceback code to feedback (and how many tokens), how long the model may answer, randomness (0 = greedy) and the test timeout.

**Returns:** An `AgentConfig`.

**Why it exists:** Each experimental condition is just a different config.

### `AttemptLog (dataclass)`

```python
@dataclass
class AttemptLog:
    attempt, n_edits, apply_errors, tests_passed, test_output_tail,
    prompt_tokens, completion_tokens, latency_s, response
```

**What it is:** A record of one attempt: how many edit blocks, any apply errors, whether tests passed (`None` if they never ran), output, cost, and the raw model reply.

**Returns:** An `AttemptLog`.

**Why it exists:** Lets us analyse *why* attempts failed (the paper found about 3 in 4 failed attempts were clean but wrong patches).

### `AgentResult (dataclass) and to_dict`

```python
@dataclass
class AgentResult: ...
    def to_dict(self) -> dict
```

**What it is:** Everything about one solved (or failed) task: `resolved`, attempts used, final patch, context size, which chunks were retrieved, `file_hit`/`symbol_hit`, token counts, model and wall time, validation output, any error, and the list of `AttemptLog`s. `to_dict` converts it (including nested logs) to a plain dictionary.

**Returns:** `to_dict` returns a dict.

**Why it exists:** Plain dictionaries can be saved as one JSON line per task in `results.jsonl`.

### `_localization`

```python
def _localization(chunks, index, gold) -> tuple[bool | None, bool | None]
```

**What it is:** Measures retrieval quality. *file hit* = some chunk is from the gold file. *symbol hit* = every gold function is fully present, either as its own chunk or inside a whole-file chunk (`covers`). Returns `(None, None)` when no gold is known.

**Inputs:**
- `chunks`: retrieved code.
- `index`: repo index.
- `gold`: gold symbol ids.

**Returns:** `(file_hit, symbol_hit)`.

**Why it exists:** Separates *did we find the bug?* from *did we fix it?*, which is the core of the paper's analysis.

### `Agent.__init__`

```python
def __init__(self, model: ModelClient, config: AgentConfig | None = None)
```

**What it is:** Stores the model and the config (default config if none is given).

**Inputs:**
- `model`: any client.
- `config`: settings.

**Returns:** An `Agent`.

**Why it exists:** —

### `Agent.solve`

```python
def solve(self, repo_dir, issue, test_cmd, feedback_cmd=None, gold=None,
          task_id='task', condition='') -> AgentResult
```

**What it is:** Runs the full loop on one task. Step by step:

**Inputs:**
- `repo_dir`: working copy with the bug.
- `issue`: bug report.
- `test_cmd`: command that decides success.
- `feedback_cmd`: command used during self-correction (defaults to `test_cmd`).
- `gold`: gold ids, only for measuring hits.
- `task_id / condition`: labels for the results.

**Returns:** An `AgentResult`.

**Why it exists:** This is the agent.

1. **Index** the repo (`RepoIndex.build`) and **retrieve** chunks for the issue within `context_budget`.
2. Measure `file_hit`/`symbol_hit`, and remember which symbols were shown.
3. Send `initial_prompt` to the model. For each attempt (up to `max_attempts`):
   - `parse_edits` → `apply_edits`. If that fails, send `no_edits_feedback` or `apply_error_feedback` and try again.
   - If it applied, `run_tests`. **Pass → stop.** Last attempt → keep this patch and stop.
   - Otherwise **`revert`**, collect traceback symbols not shown before (if `traceback_retrieval`), and send `test_failure_feedback`.
4. Finally run `test_cmd` once more as **validation**; `resolved` is that verdict.
5. Model errors (out of memory, network) are recorded in `error` instead of crashing a long experiment.

**Demo 1: self-correction.** A fake model first makes a *wrong* fix, then the right one. Watch the agent revert, show the failing test, and succeed on attempt 2.

In [23]:
from ast_agent.agent import Agent, AgentConfig
from ast_agent.mutate import apply_bug

work = fresh_copy()
apply_bug(work, task["bug"])            # plant minilib-002: median divides by 3
wrong = "I think the middle index is wrong.\n\n" + edit_block("minilib/stats.py", "    mid = n // 2", "    mid = (n - 1) // 2")
right = reply                            # the correct fix from Section 6

seen = []                                # we record every message history the model receives
def fake_model(messages):
    seen.append([dict(m) for m in messages])
    return wrong if len(seen) == 1 else right
model = ScriptedClient(fake_model)
agent = Agent(model, AgentConfig(retriever="ast_graph", max_attempts=3))
result = agent.solve(work, task["issue"], task["test_cmd"], gold=task["gold"], task_id=task["id"])

print("resolved:", result.resolved, "| attempts used:", result.attempts_used,
      "| file hit:", result.file_hit, "| symbol hit:", result.symbol_hit)
for a in result.attempts:
    print(f"  attempt {a.attempt}: edits={a.n_edits} errors={a.apply_errors} tests_passed={a.tests_passed}")
print("\nFinal patch:\n" + result.patch)

resolved: True | attempts used: 2 | file hit: True | symbol hit: True
  attempt 1: edits=1 errors=[] tests_passed=False
  attempt 2: edits=1 errors=[] tests_passed=True

Final patch:
--- a/minilib/stats.py
+++ b/minilib/stats.py
@@ -23,7 +23,7 @@
     mid = n // 2
     if n % 2 == 1:
         return data[mid]
-    return (data[mid - 1] + data[mid]) / 3
+    return (data[mid - 1] + data[mid]) / 2
 
 
 def variance(values):



Here is the **feedback message** the agent sent after attempt 1. This is what the model saw before it got it right:

In [24]:
feedback = seen[1][-1]["content"]        # last message in the 2nd call = the self-correction feedback
print(feedback[:1800])

Your patch applied cleanly but the tests still fail.

## Your previous patch
```diff
--- a/minilib/stats.py
+++ b/minilib/stats.py
@@ -20,7 +20,7 @@
     n = len(data)
     if n == 0:
         raise ValueError("median() of empty sequence")
-    mid = n // 2
+    mid = (n - 1) // 2
     if n % 2 == 1:
         return data[mid]
     return (data[mid - 1] + data[mid]) / 3
```

## Test output (tail)
```
.F..........                                                             [100%]
=================================== FAILURES ===================================
_____________________________ test_median_odd_even _____________________________
tests/test_minilib.py:14: in test_median_odd_even
    assert stats.median([4, 1, 3, 2]) == 2.5
E   assert 1.0 == 2.5
E    +  where 1.0 = <function median at 0x7f2622071ee0>([4, 1, 3, 2])
E    +    where <function median at 0x7f2622071ee0> = stats.median
=========================== short test summary info ============================
FAILED tests/test_mi

**Demo 2: the same task with no self-correction** (`max_attempts=1`). The wrong patch is final, so the task fails.

In [25]:
work = fresh_copy(); apply_bug(work, task["bug"])
r1 = Agent(ScriptedClient([wrong, right]), AgentConfig(max_attempts=1)).solve(
    work, task["issue"], task["test_cmd"], gold=task["gold"])
print("resolved:", r1.resolved, "| attempts used:", r1.attempts_used)

resolved: False | attempts used: 1


## 9. `mutate.py`: building the benchmark by planting bugs

To measure an agent fairly we need many bugs whose location we know. This module plants **one tiny bug at a time** inside a function, runs the tests, and keeps the bug only if the tests catch it. This is the idea behind *mutation testing* [DeMillo et al., 1978].

The mutation operators are defined as small dictionaries:

- **`COMPARE_SWAP`**: `<`↔`<=`, `>`↔`>=`, `==`↔`!=`
- **`BINOP_SWAP`**: `+`↔`-`, `//`→`/`
- **`BOOL_SWAP`**: `and`↔`or`
- plus removing `not` (*negation*) and changing a small integer `0, 1, 2` to `n+1` (*constant*)

**`IGNORE_DIRS`** lists folders not copied into working copies (`.git`, caches, virtual environments).

### `Mutation (dataclass)`

```python
@dataclass
class Mutation:
    path, start, end, replacement, lineno, kind
```

**What it is:** One candidate bug: replace the characters `start:end` of a file with `replacement`. `lineno` is the line and `kind` is the operator type.

**Returns:** A `Mutation`.

**Why it exists:** Character positions make the change exact, without touching anything else on the line.

### `_offsets`

```python
def _offsets(text: str) -> to_idx(lineno, col_bytes) -> int
```

**What it is:** Returns a small helper function `to_idx` that converts the AST's *(line, byte column)* positions into a character index in the text. It handles non-ASCII characters correctly, because AST columns count UTF-8 bytes, not characters.

**Inputs:**
- `text`: file source.

**Returns:** A function.

**Why it exists:** Needed to cut and replace exact operators in the source text.

### `_top_functions`

```python
def _top_functions(tree) -> iterator of function nodes
```

**What it is:** Yields every function and method that is not nested inside another function, the same set the indexer creates symbols for.

**Inputs:**
- `tree`: a parsed module.

**Returns:** A generator of function nodes.

**Why it exists:** Every planted bug then has exactly one matching gold symbol.

### `candidate_mutations`

```python
def candidate_mutations(rel: str, text: str) -> list[Mutation]
```

**What it is:** Walks every top-level function body and lists **every** place where an operator could be swapped. The inner helper `between(a, b, expected, new, kind, node)` finds the text between two sub-expressions (e.g. between `x` and `y` in `x < y`), checks that it really is the expected operator, and records the swap.

**Inputs:**
- `rel`: file path.
- `text`: file source.

**Returns:** A list of `Mutation`s.

**Why it exists:** Produces the pool of possible bugs; boltons gave 1,482 candidates.

In [26]:
from ast_agent.mutate import candidate_mutations
src = index.files["minilib/stats.py"]
cands = candidate_mutations("minilib/stats.py", src)
print(len(cands), "candidates in stats.py; first five:")
for m in cands[:5]:
    print(f"  line {m.lineno:3d} {m.kind:9s} {src[m.start:m.end]!r:8s} -> {m.replacement!r}")

18 candidates in stats.py; first five:
  line  37 compare   ' > '    -> ' >= '
  line  39 compare   ' < '    -> ' <= '
  line  41 compare   ' > '    -> ' >= '
  line  32 arith     ' - '    -> ' + '
  line  32 constant  '2'      -> '3'


### `_snippets`

```python
def _snippets(original, mutated, lineno, max_extra=6) -> (str, str) | None
```

**What it is:** Finds the smallest window of lines around the bug that appears exactly once in both the clean and the buggy file, growing the window by one line at a time (up to 6).

**Inputs:**
- `original`: clean text.
- `mutated`: buggy text.
- `lineno`: bug line.

**Returns:** `(original_snippet, buggy_snippet)` or `None`.

**Why it exists:** Tasks store only these two snippets instead of a copy of the repository; the snippets must be unique to apply the bug and its fix reliably.

### `issue_from_output`

```python
def issue_from_output(output: str) -> str
```

**What it is:** Writes the bug report from pytest output: the names of up to 5 `FAILED`/`ERROR` tests with their short error messages, plus 'Find and fix the bug... The tests are correct.' If no such lines exist, it uses the last 15 lines of output.

**Inputs:**
- `output`: test output.

**Returns:** Issue text.

**Why it exists:** The issue **never mentions the mutation**, so the agent really has to find the bug.

In [27]:
from ast_agent.mutate import issue_from_output
print(issue_from_output(broken.output))

The following tests fail on the current code:
- tests/test_minilib.py::test_median_odd_even - assert 1.666666666666666...

Find and fix the bug in the library code. The tests are correct.


### `apply_bug`

```python
def apply_bug(repo_dir: str, bug: dict) -> None
```

**What it is:** Plants a recorded bug into a working copy by replacing `original_snippet` with `buggy_snippet`. It raises an error if the snippet is missing or not unique, which would mean the clean repository changed.

**Inputs:**
- `repo_dir`: working copy.
- `bug`: the task's bug record.

**Returns:** Nothing.

**Why it exists:** Every experiment starts from a clean copy and plants exactly the same bug.

### `generate`

```python
def generate(repo, test_cmd, out_dir, max_tasks, seed, per_symbol=2, timeout=120, prefix=None) -> list[dict]
```

**What it is:** The benchmark builder. (1) Copies the repo to a temporary folder and checks the clean tests pass. (2) Collects candidate mutations from all non-test files and shuffles them with a fixed `seed` (reproducible). (3) For each candidate: skip if its function already has `per_symbol` bugs; skip if it breaks syntax or has no unique snippet; write it, run the tests, restore the file. (4) **Keep** it only if the tests fail normally (not a timeout, not a collection error). (5) Each kept bug is appended to `tasks.jsonl` immediately (so a crash loses nothing) with its id, issue text, test command, gold symbol and bug record.

**Inputs:**
- `repo`: clean repo.
- `test_cmd`: test command.
- `out_dir`: output folder.
- `max_tasks`: how many to keep (50).
- `seed`: shuffle seed (0).
- `per_symbol`: max bugs per function (2).
- `timeout`: seconds per test run.
- `prefix`: task id prefix.

**Returns:** The list of task dicts.

**Why it exists:** Builds a fair, reproducible benchmark from any pytest-tested Python library in one command.

**Demo:** build a 3-task benchmark from the sample library (this runs the test suite a few times, so it takes a few seconds).

In [28]:
from ast_agent.mutate import generate
out_dir = tempfile.mkdtemp(prefix="bench_")
new_tasks = generate(SAMPLE_REPO, "python -m pytest -q --tb=short", out_dir, max_tasks=3, seed=0)
print()
print(json.dumps(new_tasks[0], indent=2)[:700])

48 candidate mutations; testing until 3 are kept...
  kept repo-001: compare in minilib/shapes.py::Rectangle.overlaps
  kept repo-002: constant in minilib/stats.py::median
  kept repo-003: compare in minilib/stats.py::mean
Wrote 3 tasks to /tmp/…/tasks.jsonl

{
  "id": "repo-001",
  "repo": "../..gemma-ast-agent/benchmarks/sample/repo",
  "issue": "The following tests fail on the current code:\n- tests/test_minilib.py::test_overlaps - AssertionError: assert False\n\nFind and fix the bug in the library code. The tests are correct.",
  "test_cmd": "python -m pytest -q --tb=short",
  "gold": [
    "minilib/shapes.py::Rectangle.overlaps"
  ],
  "bug": {
    "path": "minilib/shapes.py",
    "line": 26,
    "kind": "compare",
    "original_snippet": "        if self.x + self.width < other.x or other.x + other.width < self.x:",
    "buggy_snippet": "        if self.x + self.width <= other.x or other.x + other.width < self.x:"
  }
}


### `main (mutate)`

```python
def main(argv=None)
```

**What it is:** The command-line entry point: reads options like `--repo`, `--out`, `--max-tasks`, `--seed` and calls `generate`.

**Inputs:**
- `argv`: command-line arguments (for testing).

**Returns:** Nothing.

**Why it exists:** Lets you run `python -m ast_agent.mutate --repo ... --out ...` from a terminal or a Kaggle cell.

## 10. `eval.py`: running experiments and computing statistics

**`CONDITIONS`** is the experiment grid. Each row changes exactly one thing:

| Condition | Retriever | Attempts | Traceback retrieval |
|---|---|---|---|
| `file` | whole files | 1 | – |
| `ast` | AST symbols | 1 | – |
| `ast_graph` | AST + call graph | 1 | – |
| `file_sc` | whole files | 3 | no |
| `ast_graph_sc` | AST + call graph | 3 | no |
| `full` | AST + call graph | 3 | yes |
| `oracle` | gold first | 1 | – |

**`DESCRIPTIONS`** holds the human-readable name of each condition for the summary table.

### `load_tasks`

```python
def load_tasks(path: str) -> list[dict]
```

**What it is:** Reads a `tasks.jsonl` file (one JSON task per line) and adds `repo_abs`, the absolute path of each task's clean repository.

**Inputs:**
- `path`: tasks file.

**Returns:** A list of task dicts.

**Why it exists:** Tasks store relative paths so a benchmark folder can be moved.

### `done_keys`

```python
def done_keys(results_path: str) -> set[tuple[str, str]]
```

**What it is:** Reads an existing `results.jsonl` and returns every `(task_id, condition)` already finished.

**Inputs:**
- `results_path`: results file.

**Returns:** A set of pairs.

**Why it exists:** **Resumable runs**: if a free Kaggle session dies after 6 hours, re-running the same command skips finished work.

### `run_one`

```python
def run_one(task, condition, model, cfg) -> dict
```

**What it is:** Runs one task under one condition: copies the clean repo to a temporary folder, plants the bug, creates the agent (or a `GoldPatchClient` when `model == "gold"`), calls `solve`, and always deletes the temporary folder afterwards.

**Inputs:**
- `task`: a task dict.
- `condition`: condition name.
- `model`: a client or `"gold"`.
- `cfg`: an `AgentConfig`.

**Returns:** The result as a dict.

**Why it exists:** Each run is isolated, so one task can never affect another.

### `wilson`

```python
def wilson(k: int, n: int, z: float = 1.96) -> tuple[float, float]
```

**What it is:** The **Wilson score interval**: a 95% confidence range for a success rate of `k` successes out of `n` [Wilson, 1927].

**Inputs:**
- `k`: successes.
- `n`: trials.
- `z`: 1.96 for 95%.

**Returns:** `(low, high)` as fractions.

**Why it exists:** With only 50 tasks, '42%' could easily be 30% or 55% by luck. The interval shows that uncertainty honestly, and unlike the simple formula it behaves well near 0% and 100%.

### `mcnemar_exact`

```python
def mcnemar_exact(b: int, c: int) -> float
```

**What it is:** The **exact McNemar test** for comparing two conditions on the *same* tasks [McNemar, 1947]. Only *discordant* tasks matter: `b` = solved by A but not B, `c` = solved by B but not A. If the conditions were equally good, each discordant task would be a coin flip; the p-value is the probability of a split at least this lopsided.

**Inputs:**
- `b`: tasks only A solved.
- `c`: tasks only B solved.

**Returns:** A two-sided p-value (smaller = stronger evidence of a real difference; below 0.05 is the usual threshold).

**Why it exists:** Paired tests are much more sensitive than comparing two percentages, because both conditions face exactly the same bugs.

In [29]:
from ast_agent.eval import wilson, mcnemar_exact
lo, hi = wilson(21, 50)
print(f"full on boltons: 21/50 = 42%, 95% CI {lo:.0%} to {hi:.0%}")
print("ast_graph vs full (0 vs 8 discordant):  p =", round(mcnemar_exact(0, 8), 4))
print("file vs full     (6 vs 13 discordant): p =", round(mcnemar_exact(6, 13), 3))

full on boltons: 21/50 = 42%, 95% CI 29% to 56%
ast_graph vs full (0 vs 8 discordant):  p = 0.0078
file vs full     (6 vs 13 discordant): p = 0.167


### `summarize`

```python
def summarize(out_dir: str, baseline: str = 'file') -> str
```

**What it is:** Reads `results.jsonl` and builds the results table: for each condition, resolved count, rate with Wilson CI, McNemar p-value against the baseline on shared tasks, average file/symbol hit, context and total tokens, attempts and model seconds. The inner helpers `pct` (format a fraction as a percentage) and `mean` (average ignoring missing values) keep it tidy. It writes `summary.md` and `summary.json`.

**Inputs:**
- `out_dir`: run folder.
- `baseline`: condition to compare against.

**Returns:** The Markdown table as a string.

**Why it exists:** One function turns raw logs into the paper's table, so nothing is copied by hand.

### `_git_commit`

```python
def _git_commit() -> str
```

**What it is:** Returns the current git commit hash of the code, or an empty string.

**Returns:** A string.

**Why it exists:** Saved in `run_meta.json`, so every result can be traced to the exact code that produced it.

### `main (eval)`

```python
def main(argv=None)
```

**What it is:** The command-line entry point. It parses options (tasks, conditions, backend `openai`/`hf`/`gold`, model, budgets, `--limit`, `--out`), creates the model client, writes `run_meta.json`, loops over conditions × tasks skipping finished pairs, appends each result to `results.jsonl` immediately, and prints the summary. `--summarize DIR` only rebuilds the table.

**Inputs:**
- `argv`: command-line arguments.

**Returns:** Nothing (prints and writes files).

**Why it exists:** This is the one command used on Kaggle.

**Demo:** the *harness check* from the paper: run the `full` condition with the **gold** backend on the 17 sample tasks. Every task must resolve. (It runs the test suite about 34 times, so allow a little while.)

In [30]:
import logging
logging.basicConfig(level=logging.WARNING)   # hide the per-attempt log lines; keep the progress and the table
from ast_agent.eval import main as eval_main
run_dir = tempfile.mkdtemp(prefix="run_")
eval_main(["--tasks", os.path.join(SAMPLE, "tasks.jsonl"), "--backend", "gold",
           "--conditions", "full", "--out", run_dir])

[1/17] full          minilib-001: RESOLVED (attempts=1, ctx=745)
[2/17] full          minilib-002: RESOLVED (attempts=1, ctx=745)
[3/17] full          minilib-003: RESOLVED (attempts=1, ctx=745)
[4/17] full          minilib-004: RESOLVED (attempts=1, ctx=745)
[5/17] full          minilib-005: RESOLVED (attempts=1, ctx=745)
[6/17] full          minilib-006: RESOLVED (attempts=1, ctx=745)
[7/17] full          minilib-007: RESOLVED (attempts=1, ctx=745)
[8/17] full          minilib-008: RESOLVED (attempts=1, ctx=745)
[9/17] full          minilib-009: RESOLVED (attempts=1, ctx=745)
[10/17] full          minilib-010: RESOLVED (attempts=1, ctx=745)
[11/17] full          minilib-011: RESOLVED (attempts=1, ctx=745)
[12/17] full          minilib-012: RESOLVED (attempts=1, ctx=745)
[13/17] full          minilib-013: RESOLVED (attempts=1, ctx=745)
[14/17] full          minilib-014: RESOLVED (attempts=1, ctx=745)
[15/17] full          minilib-015: RESOLVED (attempts=1, ctx=745)
[16/17] full       

## 11. `results/analyze.py`: regenerating every number in the paper

The Kaggle runs produced large per-attempt logs. Their essential content (which task was solved by which condition and model, plus summary costs) is stored in **`results/kaggle_runs.json`**. `analyze.py` reads that file and prints every table, paired test, operator breakdown and the second-library results, so anyone can check the paper's numbers without a GPU.

Its helpers: `solved(model, cond)` returns the set of task numbers solved, and `pct(x)` formats a percentage. The rest of the script prints the tables.

In [31]:
out = subprocess.run([sys.executable, os.path.join(ROOT, "results", "analyze.py")], capture_output=True, text=True)
print(out.stdout[:2500])

# Analysis of the Kaggle runs

Generated by `results/analyze.py` from `results/kaggle_runs.json`.

## Resolve rates (n = 50, Wilson 95% CI)

| Condition | Gemma 4 E2B | Gemma 4 E4B | Symbol hit |
|---|---|---|---|
| file | 2/50 = 4% [1%-13%] | 14/50 = 28% [17%-42%] | 74% |
| ast_graph | 1/50 = 2% [0%-10%] | 13/50 = 26% [16%-40%] | 88% |
| ast_graph_sc | 4/50 = 8% [3%-19%] | 19/50 = 38% [26%-52%] | 88% |
| full | 5/50 = 10% [4%-21%] | 21/50 = 42% [29%-56%] | 88% |
| oracle | 3/50 = 6% [2%-16%] | 17/50 = 34% [22%-48%] | 100% |

## Paired comparisons (exact two-sided McNemar)

| A -> B | Model | only A | only B | p |
|---|---|---|---|---|
| file -> ast_graph | Gemma 4 E2B | 2 | 1 | 1.0000 |
| file -> ast_graph | Gemma 4 E4B | 7 | 6 | 1.0000 |
| ast_graph -> ast_graph_sc | Gemma 4 E2B | 0 | 3 | 0.2500 |
| ast_graph -> ast_graph_sc | Gemma 4 E4B | 0 | 6 | 0.0312 |
| ast_graph_sc -> full | Gemma 4 E2B | 0 | 1 | 1.0000 |
| ast_graph_sc -> full | Gemma 4 E4B | 1 | 3 | 0.6250 |
| ast_graph -> f

## 12. Running with a real Gemma 4 model

Everything above used fake models. To reproduce the paper with Gemma 4, use **`notebooks/run_kaggle.ipynb`** on Kaggle (Accelerator: GPU T4 x2, Internet: on, add the *Gemma 4* model with framework *Transformers*). Its main command is:

```bash
python -m ast_agent.eval --tasks benchmarks/boltons/tasks.jsonl --backend hf \
  --model "$MODEL_PATH" --dtype float16 --max-new-tokens 1024 \
  --conditions file,ast_graph,ast_graph_sc,full,oracle --out results/e4b
```

On a laptop with **Ollama** you can try a few sample tasks:

```bash
ollama pull <a small gemma-4 tag>
python -m ast_agent.eval --tasks benchmarks/sample/tasks.jsonl --backend openai \
  --base-url http://localhost:11434/v1 --model <tag> --limit 3 --out results/dev
```

Or from Python:

```python
from ast_agent.models import OpenAICompatibleClient
model = OpenAICompatibleClient("http://localhost:11434/v1", "<tag>")
agent = Agent(model, AgentConfig())          # the paper's 'full' configuration
```

**What the paper found** (Gemma 4 E4B, 50 planted bugs per library):

| | whole files | AST retrieval | + test feedback (full) | oracle |
|---|---|---|---|---|
| boltons | 28% | 26% | **42%** | 34% |
| more-itertools | 2% | 22% | **36%** | 22% |

Retrieval matters when files are too big for the context (more-itertools). Test feedback gave a significant gain on both libraries, and on more-itertools it beat even the oracle by a significant margin.

## 13. Cheat sheet: every function in one table

| Module | Function / class | One-line purpose |
|---|---|---|
| indexer | `is_test_path` | Is this file a test? |
| indexer | `Symbol`, `.short_name` | One indexed piece of code; its last name part |
| indexer | `_called_names` | Names a function calls (for the call graph) |
| indexer | `_start_line` | First line including decorators |
| indexer | `_signature_lines` | A function's header only |
| indexer | `_Visitor` (`_segment`, `_add_function`, `visit_*`) | Walks the syntax tree and creates symbols |
| indexer | `RepoIndex` (`build`, `_parse`) | Index a whole repository |
| indexer | `callees`, `callers` | Call-graph neighbours |
| indexer | `symbol_at` | Which function contains this line? |
| retrieval | `estimate_tokens` | ~4 characters per token |
| retrieval | `tokenize` | Identifier-aware search words |
| retrieval | `BM25` (`scores`) | Search-engine relevance scores |
| retrieval | `ContextChunk` (`tokens`, `covers`) | One piece of prompt code; its size; does it contain a range? |
| retrieval | `traceback_frames`, `traceback_symbols` | Error message → (file, line) → functions |
| retrieval | `Retriever`, `FileRetriever` | Base class; whole-file baseline |
| retrieval | `ASTRetriever` (`rank`, `retrieve`) | Symbol ranking + call-graph expansion |
| retrieval | `symbol_chunk`, `pack_symbols` | Symbol → chunk; fill the budget greedily |
| retrieval | `OracleRetriever`, `make_retriever` | Cheating upper bound; name → retriever |
| prompts | `format_chunks`, `format_file_list`, `initial_prompt` | Build the first message |
| prompts | `no_edits_feedback`, `apply_error_feedback`, `test_failure_feedback` | Feedback messages for retries |
| models | `Generation`, `ModelClient`, `_prompt_estimate` | Reply record; interface; token fallback |
| models | `OpenAICompatibleClient`, `TransformersClient` | Real Gemma backends (server / in-process) |
| models | `ScriptedClient`, `GoldPatchClient` | Fake model; perfect-fix model for checks |
| edits | `Edit`, `ApplyResult` (`diff`) | Parsed edit; outcome and diff |
| edits | `_clean_path`, `parse_edits` | Reply text → edits |
| edits | `_find_unique`, `_indent`, `_fuzzy_replace` | Exact and indentation-tolerant matching |
| edits | `_is_protected`, `apply_edits`, `revert` | Safe atomic apply; undo |
| runner | `TestResult`, `_kill_group`, `run_tests` | Run tests in isolation with a hard timeout |
| agent | `AgentConfig`, `AttemptLog`, `AgentResult` (`to_dict`) | Settings and records |
| agent | `_localization` | File hit / symbol hit |
| agent | `Agent.solve` | The retrieve → patch → test → retry loop |
| mutate | `Mutation`, `_offsets`, `_top_functions` | Bug record and position helpers |
| mutate | `candidate_mutations` (`between`) | All possible single-operator bugs |
| mutate | `_snippets`, `issue_from_output`, `apply_bug` | Unique anchors; bug report text; plant a bug |
| mutate | `generate`, `main` | Build a benchmark |
| eval | `load_tasks`, `done_keys`, `run_one` | Load tasks; resume; run one isolated task |
| eval | `wilson`, `mcnemar_exact` | Confidence interval; paired significance test |
| eval | `summarize` (`pct`, `mean`), `_git_commit`, `main` | Results table; provenance; command line |
| analyze | `solved`, `pct` | Regenerate the paper's numbers |

### Try it yourself

1. In Section 3, change the budget from 120 to 60 tokens. Which retriever still finds `median`?
2. In Section 8, give the fake model three wrong answers. What does `result.resolved` become, and how many attempts are used?
3. In Section 9, change `seed=0` to `seed=1`. Which bugs are planted now?
4. Use `mcnemar_exact` to check the more-itertools result: `ast_graph` vs `full` was 0 vs 7.

**References:** Jimenez et al., SWE-bench (ICLR 2024) · Xia et al., Agentless (2024) · Chen et al., Self-Debugging (ICLR 2024) · Shinn et al., Reflexion (NeurIPS 2023) · Robertson & Zaragoza, BM25 (2009) · DeMillo et al., mutation testing (1978) · McNemar (1947) · Wilson (1927).